# Keras CNN MNIST

使用卷積神經網路（CNN）進行 MNIST 手寫數字辨識。

In [ ]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten

# 建立 CNN 順序模型
model = Sequential()
model.add(Conv2D(filters=16, kernel_size=(5, 5), input_shape=(28, 28, 1), activation='relu')) #新增卷積層：16 個 5x5 的濾鏡，輸入大小為 28x28x1 (單通道灰階圖)，使用 Relu
model.add(MaxPooling2D(pool_size=(2, 2))) #新增最大池化層：池化窗口大小為 2x2
model.add(Flatten()) #壓平層：將多維的特徵圖轉為一維向量
model.add(Dense(10, activation='softmax')) #輸出層：10 個分類輸出

#顯示網路結構摘要
model.summary()

#編譯模型：使用 categorical_crossentropy 損失函數
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

#載入數據集
mnist = keras.datasets.mnist
(train_data, train_label), (test_data, test_label) = mnist.load_data()
test_label_original = test_label # 備份原始數值型標籤用於後續交叉表分析

#將標籤轉換為 One-Hot 編碼 (例如：5 變為 [0,0,0,0,0,1,0,0,0,0])
train_label = keras.utils.to_categorical(train_label)
test_label = keras.utils.to_categorical(test_label)

#訓練模型，並儲存訓練歷程
train_history = model.fit(train_data, train_label, epochs=10, batch_size=512)

#評估測試集表現
score = model.evaluate(test_data, test_label)
print("測試集準確度:", score[1])

#繪製訓練時的準確度與損失歷史圖表
import matplotlib.pyplot as plt
import numpy as np

def show_train_history(train_history, train, validation):
    plt.plot(train_history.history[train])
    plt.plot(train_history.history[validation])
    plt.title('Train History')
    plt.ylabel(train)
    plt.xlabel('Epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()

show_train_history(train_history, 'accuracy', 'accuracy')
show_train_history(train_history, 'loss', 'loss')

#預測並繪製混淆矩陣 (Confusion Matrix)
import pandas as pd
prediction = model.predict(test_data)
prediction_label = np.argmax(prediction, axis=1) # 找出機率最大的類別索引
print(test_label.shape)
# 使用 pandas.crosstab 顯示預測值與實際標籤的對照矩陣
pd.crosstab(test_label_original, prediction_label, rownames=['實際值'], colnames=['預測值'])
